# このノートブックについて
OllamaとDeepEvalの`GEval`で、参照回答に基づく生成文の正確性を評価する学習用サンプルです。モデルは**利用者のPCではなく、Google ColabのGPUランタイム内**で動きます。外部のモデルAPIは使用しません。Google Driveのマウントは任意です。このNotebook専用の新しいランタイムを使用してください。

標準は固定した`evaluation_steps`です。評価基準を自由に変える場合は、フォームの`evaluation_mode`を`criteria`に切り替えてください。生成された手順は同じメトリック内で再利用され、画面に表示されます。設定を変更したら、設定セル以降を再実行します。

1. 評価用の依存関係をインストールし、必要ならDriveをマウントする。
2. 公式スクリプトの取得元と内容を確認し、Ollamaをインストールする。
3. ループバック限定のサーバーを起動し、APIの準備完了を待つ。
4. `llama3.2`を取得し、モデルの存在と構造化応答を確認する。
5. タイムアウト、モデル、評価手順を設定する。
6. TSVまたは単一例を評価し、終了時にサーバーを停止する。

ColabのGPUランタイムを選択してください。CUDAドライバーは再インストールせず、`LD_LIBRARY_PATH`も変更しません。依存インストール後に再起動を求められたら再起動し、上から実行し直してください。

## 評価値の意味と限界
これは原著者実装の再現ではありません。原論文はGPT-4等を使い、出力確率による重み付き採点を提案しています（GPT-4では当時確率を取得できず、複数サンプルから推定）。要約のcoherence/consistency/relevanceは1〜5、fluencyは1〜3で、全項目が同じ尺度ではありません。
DeepEval 4.3.2のOllama経路ではlog probabilitiesを使わず、構造化応答の通常スコア（標準0〜10）を0〜1に正規化します。原論文の性能や人手評価との相関は保証しません。`expected_output`との整合性は外部情報による事実検証ではなく、参照回答・基準・モデルに依存します。温度0でも完全な決定性は保証されません。

## 出典と利用条件
- [Liu et al. (2023), G-Eval](https://aclanthology.org/2023.emnlp-main.153/) / [原著者実装（MIT）](https://github.com/nlpyang/geval)
- [DeepEval GEval公式資料](https://deepeval.com/docs/metrics-llm-evals) / [ソース（Apache 2.0）](https://github.com/confident-ai/deepeval)
- [Ollama公式資料](https://docs.ollama.com/linux) / [Llama 3.2ライセンス](https://github.com/meta-llama/llama-models/blob/main/models/llama3_2/LICENSE)

Built with Llama. Llama 3.2には独自の利用条件とAcceptable Use Policyがあります。このリポジトリはモデル重みや依存ライブラリ本体を同梱しません。第三者の文章・画像・コードの条件はそれぞれに従ってください。


In [ ]:
#@title 必要なライブラリを評価用の依存環境へインストールする
import importlib
from pathlib import Path
import subprocess
import sys

# Colab本体とDeepEvalのclickの要求が異なるため、既存パッケージを置換しません。
evaluation_env = Path('/content/geval-venv')
evaluation_python = evaluation_env / 'bin/python'
if not evaluation_python.exists():
    subprocess.run([sys.executable, '-m', 'venv', '--without-pip', str(evaluation_env)],
                   check=True)
requirements = ['deepeval==4.3.2', 'ollama==0.6.3', 'pandas==2.2.3',
                'httpx==0.28.1', 'pydantic==2.14.2']
subprocess.run([sys.executable, '-m', 'pip', '--python', str(evaluation_python),
                'install', *requirements], check=True)
subprocess.run([sys.executable, '-m', 'pip', '--python', str(evaluation_python),
                'check'], check=True)
evaluation_site = subprocess.check_output(
    [str(evaluation_python), '-c', 'import sysconfig; print(sysconfig.get_path("purelib"))'],
    text=True,
).strip()
loaded_elsewhere = [name for name in ('deepeval', 'ollama', 'pandas', 'pydantic', 'httpx', 'click')
                    if name in sys.modules and getattr(sys.modules[name], '__file__', None)
                    and not Path(sys.modules[name].__file__).resolve().is_relative_to(
                        Path(evaluation_site).resolve())]
if loaded_elsewhere:
    raise RuntimeError('別環境のライブラリが読み込み済みです。ランタイムを再起動し、'
                       'このインストールセルを最初に実行してください: '
                       + ', '.join(loaded_elsewhere))
# 既に読み込まれた推移依存も、実際のバージョンで要求を満たすか確認します。
from importlib import metadata
from packaging.requirements import Requirement
normalize_name = lambda name: name.lower().replace('_', '-').replace('.', '-')
evaluation_distributions = list(metadata.distributions(path=[evaluation_site]))
evaluation_versions = {normalize_name(d.metadata['Name']): d.version
                       for d in evaluation_distributions}
host_versions = {normalize_name(d.metadata['Name']): d.version
                 for d in metadata.distributions(path=[p for p in sys.path
                                                      if p != evaluation_site])}
effective_versions = dict(evaluation_versions)
package_names = metadata.packages_distributions()
for module_name, module in list(sys.modules.items()):
    module_file = getattr(module, '__file__', None)
    if module_file and not Path(module_file).resolve().is_relative_to(Path(evaluation_site)):
        for distribution_name in package_names.get(module_name.split('.')[0], []):
            name = normalize_name(distribution_name)
            if name in evaluation_versions and name in host_versions:
                effective_versions[name] = host_versions[name]
conflicts = []
for distribution in evaluation_distributions:
    for requirement_text in distribution.requires or []:
        requirement = Requirement(requirement_text)
        if requirement.marker and not requirement.marker.evaluate({'extra': ''}):
            continue
        name = normalize_name(requirement.name)
        actual_version = effective_versions.get(name)
        if actual_version is None or not requirement.specifier.contains(actual_version):
            conflicts.append(f'{distribution.metadata["Name"]}: {requirement_text} '
                             f'(loaded: {actual_version})')
if conflicts:
    raise RuntimeError('読み込み済みの依存が評価用の要求と競合しています。'
                       '新しいランタイムでこのセルを最初に実行してください: '
                       + '; '.join(conflicts))
if evaluation_site not in sys.path:
    sys.path.insert(0, evaluation_site)
importlib.invalidate_caches()
print('評価用の依存環境:', evaluation_site)
# 以前の評価用ライブラリをimport済みの場合は、ランタイムを再起動してから
# このセルを最初に実行してください。


In [ ]:
#@title Google Driveをマウントする（必要な場合だけ実行）
from google.colab import drive
drive.mount('/content/drive/')


In [ ]:
#@title Ollamaをインストールする（Colab Linux専用）
# 公式スクリプトはランタイム内のシステムを変更します。
# 取得後に内容を確認してから、次のセルを実行してください。
import subprocess
import shutil
from pathlib import Path

# 公式インストーラーの展開に必要なコマンドだけを補います。
if shutil.which('zstd') is None:
    for command in (['apt-get', 'update', '-qq'], ['apt-get', 'install', '-y', 'zstd']):
        setup = subprocess.run(command, capture_output=True, text=True)
        print(setup.stdout[-2000:], setup.stderr[-2000:])
        setup.check_returncode()

installer = Path('/content/ollama-install.sh')
subprocess.run(
    ['curl', '-fSL', 'https://ollama.com/install.sh', '-o', str(installer)],
    check=True,
)
print(installer.read_text())


In [ ]:
#@title 確認した公式インストーラーを実行し、Ollamaを起動する
import atexit
import os
import time
import subprocess
import urllib.request
import urllib.error
from pathlib import Path

installation = subprocess.run(['sh', str(installer)], capture_output=True, text=True)
print(installation.stdout[-4000:], installation.stderr[-4000:])
installation.check_returncode()
# Colabが提供するCUDAドライバーとLD_LIBRARY_PATHをそのまま使用します。
OLLAMA_BASE_URL = 'http://127.0.0.1:11434'
OLLAMA_LOG = Path('/content/ollama.log')
ollama_process = globals().get('ollama_process')

def wait_for_ollama(process, base_url, timeout=60):
    deadline = time.monotonic() + timeout
    last_error = None
    opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))
    while time.monotonic() < deadline:
        if process.poll() is not None:
            raise RuntimeError(f'Ollamaが終了しました。ログを確認: {OLLAMA_LOG}')
        try:
            with opener.open(base_url + '/api/version', timeout=1) as response:
                if response.status == 200:
                    return
        except (urllib.error.URLError, TimeoutError, OSError) as exc:
            last_error = exc
        time.sleep(0.25)
    raise TimeoutError(f'Ollamaの起動待ちが期限切れです。ログ: {OLLAMA_LOG}') from last_error

def stop_ollama_server():
    # このNotebookが起動したプロセスだけを停止します。
    process = globals().get('ollama_process')
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=5)

# 起動直後の中断でも終了処理を失わないよう、起動より前に登録します。
atexit.unregister(stop_ollama_server)
atexit.register(stop_ollama_server)
try:
    if ollama_process is None or ollama_process.poll() is not None:
        # 別のサーバーに誤って接続しないよう、既存リスナーがあれば処理を停止します。
        import socket
        with socket.socket() as probe:
            if probe.connect_ex(('127.0.0.1', 11434)) == 0:
                raise RuntimeError('ポート11434は使用中です。既存サーバーを確認してください。')
        server_env = dict(os.environ, OLLAMA_HOST='127.0.0.1:11434', OLLAMA_NO_CLOUD='1')
        with OLLAMA_LOG.open('a', encoding='utf-8') as log:
            ollama_process = subprocess.Popen(
                ['ollama', 'serve'], env=server_env, stdout=log, stderr=subprocess.STDOUT,
            )
    wait_for_ollama(ollama_process, OLLAMA_BASE_URL)
except BaseException:
    # Colabの停止ボタンによるKeyboardInterruptでも停止し、再送出します。
    stop_ollama_server()
    raise
print('Ollama API ready:', OLLAMA_BASE_URL)


In [ ]:
#@title セッション終了時の停止処理を登録する
import atexit
atexit.unregister(stop_ollama_server)
atexit.register(stop_ollama_server)
# 通常終了時には、最後の「ランタイムの切断」セルでも明示的に停止します。


In [ ]:
#@title llama3.2モデルをダウンロードする
MODEL_NAME = 'llama3.2'
subprocess.run(['ollama', 'pull', MODEL_NAME], check=True,
               env=dict(os.environ, OLLAMA_HOST=OLLAMA_BASE_URL))
subprocess.run(['ollama', '--version'], check=True)


In [ ]:
#@title モデルの存在と構造化応答を確認する
import json
import ollama

client = ollama.Client(host=OLLAMA_BASE_URL, timeout=120, trust_env=False)
client.show(MODEL_NAME)  # モデル未取得・接続失敗はここで停止します。
response = client.chat(
    model=MODEL_NAME,
    messages=[{'role': 'user', 'content': 'Return a JSON object with ok set to true.'}],
    format={'type': 'object', 'properties': {'ok': {'type': 'boolean'}},
            'required': ['ok']},
    options={'temperature': 0},
)
if json.loads(response.message.content).get('ok') is not True:
    raise ValueError('モデルが期待する構造化応答を返しませんでした。')
print('接続・構造化応答を確認しました。モデル情報:')
print(client.list())  # モデルのタグと識別情報を表示します。


In [ ]:
#@title DeepEvalの通信とタイムアウトを設定する（初回importより前）
import os

timeout_duration = 60  #@param {type:"number"}
if timeout_duration <= 0:
    raise ValueError('timeout_durationには正の秒数を指定してください。')
os.environ['DEEPEVAL_TELEMETRY_OPT_OUT'] = '1'
os.environ['DEEPEVAL_DISABLE_TIMEOUTS'] = '0'
os.environ['DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE'] = str(timeout_duration)
# 接続・設定エラーを長時間再試行せず、利用者へ通知します。
os.environ['DEEPEVAL_RETRY_MAX_ATTEMPTS'] = '1'


In [ ]:
#@title DeepEvalにOllamaモデルを明示的に渡す
from deepeval.config.settings import get_settings
from deepeval.models import OllamaModel

# 設定セルを再実行した場合も、キャッシュ済み設定へ反映します。
settings = get_settings()
settings.DEEPEVAL_TELEMETRY_OPT_OUT = True
settings.DEEPEVAL_DISABLE_TIMEOUTS = False
settings.DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE = float(timeout_duration)
settings.DEEPEVAL_RETRY_MAX_ATTEMPTS = 1
judge = OllamaModel(model=MODEL_NAME, base_url=OLLAMA_BASE_URL,
                    temperature=0, timeout=timeout_duration, trust_env=False)
# set-ollama CLIの永続設定や、model='llama3.2'だけの暗黙選択は使いません。


In [ ]:
#@title 評価メトリックを定義する
import math
from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams

evaluation_mode = 'fixed'  #@param ["fixed", "criteria"]
criteria = 'Evaluate the Actual Output against the Expected Output as the reference answer to Input. Do not compare the wording of Input with the Expected Output. Allow semantically equivalent answers. Score 10 for a fully correct answer and 0 for a contradictory answer.'  #@param {type:"string"}
evaluation_steps = [
    'Read the input and expected output to identify the required facts.',
    'Treat Expected Output as the reference answer. Compare Actual Output with Expected Output, allowing equivalent wording. Do not compare the wording of Input with Expected Output.',
    'Penalize contradictions and missing required facts. Do not verify external facts.',
    'Assign 10 when Actual Output agrees fully with Expected Output, 0 for a contradictory answer, and an intermediate score for a partially correct answer.',
]
if evaluation_mode not in ('fixed', 'criteria'):
    raise ValueError('evaluation_modeはfixedまたはcriteriaです。')
metric_options = ({'evaluation_steps': evaluation_steps} if evaluation_mode == 'fixed'
                  else {'criteria': criteria})
correctness_metric = GEval(
    name='Correctness',
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT,
                       SingleTurnParams.EXPECTED_OUTPUT],
    model=judge, async_mode=True, **metric_options,
)
# criteriaを変更する場合はcriteriaモードを選び、このセルを再実行します。
# 生成された手順は同じメトリック内で再利用されます。

class InvalidEvaluationResponse(ValueError):
    pass

def measure_case(metric, test_case):
    # 成功時だけ呼び出し元へ値を返します。TSV・単一例で同じ検証を使います。
    metric.score = metric.reason = None
    metric.measure(test_case, _show_indicator=False)
    steps = metric.evaluation_steps
    if (not isinstance(steps, list) or not steps
            or any(not isinstance(step, str) or not step.strip() for step in steps)):
        # 次の行ではcriteriaから再生成し、不完全な手順を再利用しません。
        metric.evaluation_steps = None
        raise InvalidEvaluationResponse('評価手順が空または不正です。')
    try:
        score = float(metric.score)
    except (TypeError, ValueError) as exc:
        raise InvalidEvaluationResponse('数値のスコアがありません。') from exc
    if not math.isfinite(score) or not 0 <= score <= 1:
        raise InvalidEvaluationResponse('評価モデルのスコアが0〜1の有限値ではありません。')
    if not isinstance(metric.reason, str) or not metric.reason.strip():
        raise InvalidEvaluationResponse('評価モデルが理由を返しませんでした。')
    return score, metric.reason


In [ ]:
#@title 評価を実行して、TSVファイルに出力する
import csv
import json
import tempfile
from pathlib import Path
import pandas as pd
import httpx
import ollama
from pydantic import ValidationError
from deepeval.test_case import LLMTestCase

input_file = '/content/sample.tsv'  #@param {type:"string"}
output_file = '/content/sample.scored.tsv'  #@param {type:"string"}
input_column = 'input_column'  #@param {type:"string"}
actual_output_column = 'actual_output_column'  #@param {type:"string"}
expected_output_column = 'expected_output_column'  #@param {type:"string"}

def exception_chain(exc):
    seen = set()
    while exc is not None and id(exc) not in seen:
        seen.add(id(exc))
        yield exc
        last_attempt = getattr(exc, 'last_attempt', None)
        exc = (last_attempt.exception() if last_attempt is not None
               else exc.__cause__ or exc.__context__)

def error_status(exc):
    # 設定・接続エラーを優先し、メッセージの部分一致で判定しません。
    chain = list(exception_chain(exc))
    for cause in chain:
        if isinstance(cause, (ConnectionError, httpx.NetworkError,
                              httpx.HTTPStatusError, ollama.ResponseError)):
            return 'fatal_error'
    if any(isinstance(cause, (TimeoutError, httpx.TimeoutException)) for cause in chain):
        return 'timeout'
    if any(isinstance(cause, (json.JSONDecodeError, ValidationError, InvalidEvaluationResponse)) for cause in chain):
        return 'invalid_response'
    return 'fatal_error'

def save_results(data, target):
    # 完成したTSVだけを置換し、書き込み途中のファイルを残さないようにします。
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', newline='',
                                         dir=target.parent, delete=False) as stream:
            temporary = Path(stream.name)
            data.to_csv(stream, sep='\t', index=False)
        temporary.replace(target)
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)

def evaluate_tsv(input_path, output_path, metric, columns):
    source, target = Path(input_path).resolve(), Path(output_path).resolve()
    if source == target:
        raise ValueError('入力と出力には別のパスを指定してください。')
    if target.exists():
        raise FileExistsError('出力先は既に存在します。別のファイル名を指定してください。')
    if not target.parent.is_dir():
        raise FileNotFoundError(f'出力先ディレクトリがありません: {target.parent}')
    with source.open(encoding='utf-8-sig', newline='') as stream:
        reader = csv.reader(stream, delimiter='\t', strict=True)
        header = next(reader, [])
        records = []
        for row_number, record in enumerate(reader, start=1):
            if len(record) > len(header):
                raise ValueError(f'TSVのデータ行{row_number}の列数がヘッダーを超えています。')
            # 不足した末尾セルと空行は欠損値として行を保持します。
            records.append(record + [''] * (len(header) - len(record)))
    if not header or len(header) != len(set(header)):
        raise ValueError('TSVには重複のない列名が必要です。')
    if len(columns) != 3 or len(set(columns)) != 3 or not all(columns):
        raise ValueError('入力・実出力・参照回答には異なる3列を指定してください。')
    missing = set(columns) - set(header)
    if missing:
        raise ValueError(f'必須列がありません: {sorted(missing)}')
    result_columns = ['Score', 'Reason', 'Status', 'Error']
    if set(result_columns) & set(header):
        raise ValueError('入力TSVに結果列が存在します。上書きしないよう名前を変更してください。')
    data = pd.DataFrame(records, columns=header, dtype=str)
    for name in result_columns:
        data[name] = ''
    data['Score'] = pd.Series([None] * len(data), dtype=object)
    data['Status'] = 'not_processed'
    shown_steps = None
    # 同じ出力先を使う並行実行も、評価開始前に排他的に拒否します。
    # 予約ファイルには全行をnot_processedとして保存しておきます。
    with target.open('x', encoding='utf-8', newline='') as reservation:
        data.to_csv(reservation, sep='\t', index=False)
    try:
        for index, row in data.iterrows():
            values = [row[name] for name in columns]
            if any(not isinstance(value, str) or not value.strip() for value in values):
                data.at[index, 'Status'] = 'invalid_input'
                data.at[index, 'Error'] = '必須値が欠損・空文字・空白です。'
                print(f'行{index + 1}: invalid_input')
                continue
            try:
                score, reason = measure_case(metric, LLMTestCase(
                    input=values[0], actual_output=values[1], expected_output=values[2],
                ))
                data.at[index, 'Score'] = score
                data.at[index, 'Reason'] = reason
                data.at[index, 'Status'] = 'success'
            except Exception as exc:
                status = error_status(exc)
                data.at[index, 'Status'] = status
                data.at[index, 'Error'] = f'{type(exc).__name__}: {exc}'
                print(f'行{index + 1}: {status}: {type(exc).__name__}: {exc}')
                if status == 'fatal_error':
                    raise
            finally:
                steps = getattr(metric, 'evaluation_steps', None)
                if steps and steps != shown_steps:
                    print('使用したevaluation_steps:', json.dumps(steps, ensure_ascii=False))
                    shown_steps = list(steps)
    finally:
        # 致命的エラー・中断時も全入力行を保持し、後続行はnot_processedで保存します。
        save_results(data, target)
    print(f'結果を保存しました: {target}')
    return data

# リポジトリのsamples/sample.tsvを/content/sample.tsvへアップロードするか、
# Drive上のパスをフォームへ指定してください。出力済みファイルは上書きしません。
results = evaluate_tsv(input_file, output_file, correctness_metric,
                       [input_column, actual_output_column, expected_output_column])
# タイムアウトはクライアント側の待機を中止します。サーバー側の推論停止は
# 保証されず、次の逐次リクエストが残存処理の影響で待つ場合があります。


In [ ]:
#@title TSVファイルを出力しない場合のコード（任意）
from deepeval.test_case import LLMTestCase

single_input = '架空の町リノの花は何色ですか。'  #@param {type:"string"}
single_actual_output = '青色です。'  #@param {type:"string"}
single_expected_output = '青色です。'  #@param {type:"string"}
score, reason = measure_case(correctness_metric, LLMTestCase(
    input=single_input, actual_output=single_actual_output,
    expected_output=single_expected_output,
))
print('Score:', score)
print('Reason:', reason)
print('evaluation_steps:', correctness_metric.evaluation_steps)


In [ ]:
#@title ランタイムの切断（全ての評価と保存が終わってから実行）
stop_ollama_server()
from google.colab import runtime
runtime.unassign()
